# 02 - Fraud Detection Model Training (v2)

XGBoost binary classification. Uses mv.run() for batch inference (no SPCS dependency).
All objects in TSHO_SWT_TOKYO_26.FRAUD / FRAUD_ML.

In [ ]:
import os
from datetime import datetime

import pandas as pd
from snowflake.snowpark import Session

DB = "TSHO_SWT_TOKYO_26"
FRAUD_SCHEMA = f"{DB}.FRAUD"
ML_SCHEMA = f"{DB}.FRAUD_ML"

connection_name = os.environ.get("SNOWFLAKE_CONNECTION_NAME", "default")
session = Session.builder.configs({"connection_name": connection_name}).create()
session.sql(f"USE SCHEMA {FRAUD_SCHEMA}").collect()
print(f"Connected: {session.get_current_role()}")

## 1. Data Preparation

In [ ]:
df = session.table("FRAUD_FEATURES").to_pandas()
print(f"FRAUD_FEATURES: {len(df)} rows")
print(f"Fraud rate: {df['IS_FRAUD'].mean():.3f}")
print(f"Class distribution:\n{df['IS_FRAUD'].value_counts()}")

In [ ]:
NUMERIC_FEATURES = [
    "AMOUNT",
    "ACCOUNT_AGE_DAYS",
    "COUNTRY_CHANGED_FLAG",
    "HIGH_RISK_MERCHANT_FLAG",
    "TXN_COUNT_1H",
    "AMOUNT_SUM_1H",
    "AMOUNT_AVG_1H",
    "TXN_COUNT_24H",
    "AMOUNT_SUM_24H",
    "AMOUNT_AVG_24H",
    "TXN_COUNT_7D",
    "AMOUNT_SUM_7D",
    "AMOUNT_AVG_7D",
]
CAT_FEATURES = ["CHANNEL", "CUSTOMER_SEGMENT", "MERCHANT_CATEGORY", "MERCHANT_RISK_LEVEL"]
TARGET = "IS_FRAUD"

df_encoded = pd.get_dummies(df[NUMERIC_FEATURES + CAT_FEATURES + [TARGET]], columns=CAT_FEATURES)
feature_cols = [c for c in df_encoded.columns if c != TARGET]
print(f"Feature columns: {len(feature_cols)}")

## 2. Train/Test Split

In [ ]:
from sklearn.model_selection import train_test_split

X = df_encoded[feature_cols].values
y = df_encoded[TARGET].values
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train: {len(X_train)} ({y_train.mean():.3f} fraud rate)")
print(f"Test:  {len(X_test)} ({y_test.mean():.3f} fraud rate)")

## 3. XGBoost Training

In [ ]:
from xgboost import XGBClassifier

scale_pos_weight = (y_train == 0).sum() / max((y_train == 1).sum(), 1)
print(f"scale_pos_weight: {scale_pos_weight:.1f}")

model = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    scale_pos_weight=scale_pos_weight,
    eval_metric="aucpr",
    random_state=42,
    use_label_encoder=False,
)
model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=20)

## 4. Evaluation

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

metrics = {
    "accuracy": accuracy_score(y_test, y_pred),
    "precision": precision_score(y_test, y_pred, zero_division=0),
    "recall": recall_score(y_test, y_pred, zero_division=0),
    "f1": f1_score(y_test, y_pred, zero_division=0),
    "roc_auc": roc_auc_score(y_test, y_proba),
}
for k, v in metrics.items():
    print(f"  {k}: {v:.4f}")

print(f"\n{classification_report(y_test, y_pred)}")
print(f"Confusion Matrix:\n{confusion_matrix(y_test, y_pred)}")

## 5. Register Model

In [ ]:
from snowflake.ml.registry import Registry

reg = Registry(session=session, database_name=DB, schema_name="FRAUD_ML")
model_version = datetime.now().strftime("v%Y%m%d_%H%M%S")

sample_input = pd.DataFrame(X_test[:5], columns=feature_cols)
sample_input = sample_input.apply(pd.to_numeric, errors="coerce")

mv = reg.log_model(
    model,
    model_name="FRAUD_DETECTION_XGBOOST",
    version_name=model_version,
    sample_input_data=sample_input,
    comment=f"XGBoost fraud detector. F1={metrics['f1']:.4f}, AUC={metrics['roc_auc']:.4f}",
    metrics=metrics,
    target_platforms=["WAREHOUSE"],
)
print(f"Model registered: FRAUD_DETECTION_XGBOOST {model_version}")


## 6. Batch Inference with mv.run()

Uses native SQL batch inference (warehouse execution).
**NOT** mv.run_batch() which requires SPCS compute pool.

In [ ]:
import snowflake.snowpark.functions as F

raw_df = session.table(f"{FRAUD_SCHEMA}.FRAUD_FEATURES")

numeric_cols = [
    "AMOUNT", "ACCOUNT_AGE_DAYS",
    "COUNTRY_CHANGED_FLAG", "HIGH_RISK_MERCHANT_FLAG",
    "TXN_COUNT_1H", "AMOUNT_SUM_1H", "AMOUNT_AVG_1H",
    "TXN_COUNT_24H", "AMOUNT_SUM_24H", "AMOUNT_AVG_24H",
    "TXN_COUNT_7D", "AMOUNT_SUM_7D", "AMOUNT_AVG_7D",
]

# One-hot encode categoricals as BOOL with lowercase names (matching pd.get_dummies output)
ohe_exprs = []
cat_values = {
    "CHANNEL": ["atm", "mobile", "pos", "web"],
    "CUSTOMER_SEGMENT": ["business", "individual", "premium", "student"],
    "MERCHANT_CATEGORY": ["clothing", "electronics", "entertainment", "fuel", "grocery", "health", "restaurant", "travel"],
    "MERCHANT_RISK_LEVEL": ["high", "low", "medium"],
}
for col_name, values in cat_values.items():
    for val in values:
        # Use quoted alias to preserve lowercase name, cast to BOOL
        alias = f"{col_name}_{val}"
        ohe_exprs.append(
            F.when(F.col(col_name) == val, F.lit(True)).otherwise(F.lit(False)).alias('"' + alias + '"')
        )

input_df = raw_df.select(
    F.col("TRANSACTION_ID"),
    F.col("CUSTOMER_ID"),
    *[F.col(c) for c in numeric_cols],
    *ohe_exprs,
)
print(f"Input rows: {input_df.count()}")
print(f"Input columns ({len(input_df.columns)}): {input_df.columns[:5]}...")

# mv.run() with IDs — IDs are passed through
result_df = mv.run(input_df, function_name="predict_proba")
print(f"Inference complete. Result columns: {result_df.columns}")
result_df.show(3)


In [ ]:
# mv.run() returns the feature columns + output columns.
# We need to rejoin with IDs. Use the input_df which has IDs + features,
# and result_df which has features + output. Since mv.run() preserves row order
# we add row_number to both and join.
from snowflake.snowpark import Window

# Find the score column (predict_proba output)
output_cols = [c for c in result_df.columns if "output_feature_1" in c.lower() or "predict_proba_1" in c.lower()]
if not output_cols:
    output_cols = [c for c in result_df.columns if "proba" in c.lower() or "output" in c.lower()]
    print(f"Available output columns: {result_df.columns}")
score_col = output_cols[-1] if output_cols else result_df.columns[-1]
print(f"Using score column: {score_col}")

# Alternative: run inference with IDs included, mv.run passes through extra columns
# Re-run with full input that includes IDs
result_with_ids = mv.run(input_df, function_name="predict_proba")

scores_df = result_with_ids.select(
    F.col("TRANSACTION_ID"),
    F.col("CUSTOMER_ID"),
    F.col(score_col).cast("FLOAT").alias("FRAUD_SCORE"),
    F.when(F.col(score_col) >= 0.5, F.lit(1)).otherwise(F.lit(0)).alias("PREDICTION"),
    F.lit(model_version).alias("MODEL_VERSION"),
    F.lit(0.5).alias("THRESHOLD"),
    F.current_timestamp().alias("SCORED_AT"),
)

session.sql(f"TRUNCATE TABLE IF EXISTS {FRAUD_SCHEMA}.FRAUD_SCORES").collect()
scores_df.write.mode("append").save_as_table(f"{FRAUD_SCHEMA}.FRAUD_SCORES")

cnt = session.sql(f"SELECT COUNT(*) AS cnt FROM {FRAUD_SCHEMA}.FRAUD_SCORES").collect()[0]["CNT"]
flagged = session.sql(f"SELECT SUM(PREDICTION) AS cnt FROM {FRAUD_SCHEMA}.FRAUD_SCORES").collect()[0]["CNT"]
print(f"FRAUD_SCORES: {cnt} rows written ({flagged} flagged)")


## 7. Create Model Monitor

Register a Model Monitor for drift detection in Snowsight Model Gateway.

In [ ]:
# Create Model Monitor via SQL
try:
    session.sql('SELECT 1').collect()
except Exception:
    from snowflake.snowpark import Session
    connection_name = os.environ.get('SNOWFLAKE_CONNECTION_NAME', 'default')
    session = Session.builder.configs({'connection_name': connection_name}).create()

# model_version uses lowercase v; Snowflake stores as uppercase
version_upper = model_version.upper()
print(f'Creating monitor for version: {version_upper}')

session.sql(f'''
CREATE OR REPLACE MODEL MONITOR {DB}.FRAUD_ML.FRAUD_MODEL_MONITOR WITH
    MODEL = {DB}.FRAUD_ML.FRAUD_DETECTION_XGBOOST
    VERSION = '{version_upper}'
    FUNCTION = 'predict'
    SOURCE = {DB}.FRAUD.PREDICTION_LOG
    WAREHOUSE = TSHO_WH_XL
    REFRESH_INTERVAL = '1 hour'
    AGGREGATION_WINDOW = '1 day'
    TIMESTAMP_COLUMN = SCORED_AT
    PREDICTION_CLASS_COLUMNS = ('PREDICTION')
    ACTUAL_CLASS_COLUMNS = ('IS_FRAUD')
''').collect()
print(f'Model Monitor created')


In [ ]:
session.close()
print("Done.")